# 🌾 Maji Ndogo Agricultural Data Analysis
**Author:** Oluwatobi Victoria Babalola | MSc Data Science  
**Portfolio:** [bbt-portfolio.vercel.app](https://bbt-portfolio.vercel.app) | **GitHub:** [TechieBbt](https://github.com/TechieBbt) | **LinkedIn:** [oluwatobi-babalola-victoria](https://www.linkedin.com/in/oluwatobi-babalola-victoria/)

---

## Project Overview

Maji Ndogo is a diverse agricultural region facing the challenge of scaling up food production through data-driven farming decisions. This project analyses a multi-table SQLite farm survey database to uncover which crops thrive under which conditions — and where automated farming systems should be deployed for maximum yield.

This is a real-world data pipeline project: raw data lives in a relational database, requires SQL joins, cleaning, and Pandas analysis before any insights can be extracted.

## Objectives
1. Ingest and join multi-table agricultural data from an SQLite database using SQL
2. Clean and validate the dataset (fix column swaps, spelling errors, invalid values)
3. Analyse crop distribution, soil fertility, and climate-geography relationships
4. Identify top-performing crops and their ideal growing conditions
5. Build reusable, well-documented analytical functions

## Dataset
The `Maji_Ndogo_farm_survey_small.db` SQLite database contains **four tables** joined on `Field_ID`:

| Table | Key Features |
|-------|--------------|
| `geographic_features` | Elevation, Latitude, Longitude, Location, Slope |
| `weather_features` | Rainfall, Min/Max/Ave Temperature |
| `soil_and_crop_features` | Soil fertility, Soil type, pH |
| `farm_management_features` | Pollution level, Plot size, Crop type, Annual yield, Standard yield |

## Tools Used
`Python` · `Pandas` · `SQLAlchemy` · `SQLite` · `Matplotlib` · `Logging` · `OOP (FieldDataProcessor)`

---
## 1. Setup & Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from sqlalchemy import create_engine, text
import warnings
warnings.filterwarnings('ignore')

# Display settings
pd.set_option('display.float_format', lambda x: f'{x:.4f}')
pd.set_option('display.max_columns', 20)
plt.rcParams.update({
    'figure.facecolor': '#f9f6f0',
    'axes.facecolor':   '#f9f6f0',
    'axes.spines.top':  False,
    'axes.spines.right':False,
    'font.family':      'DejaVu Sans',
    'axes.titleweight': 'bold',
    'axes.titlesize':   13,
})
PALETTE = ['#C8952A','#1A6B6B','#B03A2E','#2E4057','#8B5E3C','#5D6D7E','#A9CCE3','#76D7C4']

print('✅ Libraries loaded')

---
## 2. Data Ingestion — SQL to Pandas

The survey data is stored across four tables in an SQLite database. We use a single SQL `JOIN` query to merge all tables on the common key `Field_ID` — loading the full dataset into a single Pandas DataFrame in one step.

In [ ]:
# Connect to the SQLite database
engine = create_engine('sqlite:///Maji_Ndogo_farm_survey_small.db')

# Verify available tables
with engine.connect() as conn:
    result = conn.execute(text("SELECT name FROM sqlite_master WHERE type='table';"))
    tables = [row[0] for row in result]

print('📋 Tables found in database:')
for t in tables:
    print(f'   → {t}')

In [ ]:
# Join all four tables into a single DataFrame
sql_query = """
SELECT *
FROM geographic_features    AS g
JOIN weather_features       AS w ON g.Field_ID = w.Field_ID
JOIN soil_and_crop_features AS s ON g.Field_ID = s.Field_ID
JOIN farm_management_features AS f ON g.Field_ID = f.Field_ID
"""

with engine.connect() as conn:
    MD_agric_df = pd.read_sql_query(sql_query, conn)

# Drop duplicate Field_ID columns introduced by the joins
MD_agric_df.drop(columns='Field_ID', inplace=True)

print(f'✅ Data loaded successfully')
print(f'   Shape: {MD_agric_df.shape[0]:,} rows × {MD_agric_df.shape[1]} columns')
print(f'\n{MD_agric_df.head(3).to_string()}')

---
## 3. Data Cleaning & Validation

Three issues were identified in the raw data:
1. **Swapped column names** — `Rainfall` and `Min_temperature_C` headers were transposed
2. **Crop type spelling errors** — inconsistent/misspelled crop names
3. **Negative elevation values** — physically impossible; corrected to absolute values

In [ ]:
# --- Inspect before cleaning ---
print('=== Missing Values ===')
print(MD_agric_df.isnull().sum())

print('\n=== Data Types ===')
print(MD_agric_df.dtypes)

print('\n=== Crop types BEFORE cleaning ===')
print(MD_agric_df['Crop_type'].unique())

print('\n=== Elevation range BEFORE cleaning ===')
print(f'  Min: {MD_agric_df["Elevation"].min():.2f}  |  Max: {MD_agric_df["Elevation"].max():.2f}')

In [ ]:
# --- Fix 1: Swap column names (Rainfall ↔ Min_temperature_C were transposed) ---
MD_agric_df.rename(columns={
    'Rainfall': '__temp__',
    'Min_temperature_C': 'Rainfall'
}, inplace=True)
MD_agric_df.rename(columns={'__temp__': 'Min_temperature_C'}, inplace=True)
print('✅ Fix 1: Column names corrected')

# --- Fix 2: Standardise crop type spellings ---
crop_corrections = {
    'cassaval': 'cassava',
    'teaa':     'tea',
    'wheattt':  'wheat',
    'maizee':   'maize',
    'coffeee':  'coffee',
    'potatoo':  'potato',
    'bananaa':  'banana',
    'ricee':    'rice',
}
MD_agric_df['Crop_type'] = (
    MD_agric_df['Crop_type']
    .str.strip()
    .replace(crop_corrections)
)
print('✅ Fix 2: Crop type spellings corrected')

# --- Fix 3: Make elevation values non-negative ---
MD_agric_df['Elevation'] = MD_agric_df['Elevation'].abs()
print('✅ Fix 3: Negative elevation values corrected')

# --- Validation ---
print(f'\n=== Post-cleaning validation ===')
print(f'  Unique crop types ({len(MD_agric_df["Crop_type"].unique())}): {sorted(MD_agric_df["Crop_type"].unique())}')
print(f'  Elevation min: {MD_agric_df["Elevation"].min():.2f} (should be > 0)')
print(f'  Annual_yield dtype: {MD_agric_df["Annual_yield"].dtype}')

---
## 4. Exploratory Data Analysis

In [ ]:
print('=== Dataset Summary Statistics ===')
print(MD_agric_df.describe().round(3).to_string())

In [ ]:
# ── CHART 1: Distribution of crops ──
crop_counts = MD_agric_df['Crop_type'].value_counts()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

crop_counts.plot(kind='bar', ax=axes[0], color=PALETTE, edgecolor='white', alpha=0.9)
axes[0].set_title('Number of Fields per Crop Type')
axes[0].set_xlabel('')
axes[0].set_ylabel('Field Count')
axes[0].set_xticklabels(crop_counts.index, rotation=30, ha='right')

MD_agric_df.groupby('Crop_type')['Standard_yield'].mean().sort_values().plot(
    kind='barh', ax=axes[1], color=PALETTE[0], alpha=0.85
)
axes[1].set_title('Average Standard Yield by Crop Type')
axes[1].set_xlabel('Mean Standard Yield')

plt.suptitle('Crop Distribution & Performance — Maji Ndogo', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('chart1_crop_overview.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Chart 1 saved')

In [ ]:
# ── CHART 2: Pollution vs Standard Yield ──
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(
    MD_agric_df['Pollution_level'], MD_agric_df['Standard_yield'],
    alpha=0.3, s=10, color=PALETTE[2]
)
axes[0].set_title('Pollution Level vs Standard Yield')
axes[0].set_xlabel('Pollution Level')
axes[0].set_ylabel('Standard Yield')

MD_agric_df['Standard_yield'].plot(
    kind='hist', bins=30, ax=axes[1], color=PALETTE[1], alpha=0.85, edgecolor='white'
)
axes[1].set_title('Distribution of Standard Yield')
axes[1].set_xlabel('Standard Yield')
axes[1].axvline(MD_agric_df['Standard_yield'].mean(), color=PALETTE[2],
               linestyle='--', linewidth=2, label=f'Mean = {MD_agric_df["Standard_yield"].mean():.3f}')
axes[1].legend()

plt.tight_layout()
plt.savefig('chart2_yield_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 5. Analytical Functions

The following reusable functions encapsulate the core analyses. Each function is documented, testable, and designed for production use.

In [ ]:
def explore_crop_distribution(df, crop_filter):
    """
    Returns the mean Rainfall and Elevation for a given crop type.

    Parameters
    ----------
    df : pd.DataFrame  — the full agricultural dataset
    crop_filter : str  — crop type to filter on (e.g. 'tea', 'wheat')

    Returns
    -------
    tuple : (mean_rainfall, mean_elevation)

    Example
    -------
    >>> explore_crop_distribution(MD_agric_df, 'tea')
    (1534.51, 775.21)
    """
    filtered = df[df['Crop_type'] == crop_filter]
    mean_rainfall  = filtered['Rainfall'].mean()
    mean_elevation = filtered['Elevation'].mean()
    return (mean_rainfall, mean_elevation)


# --- Test ---
print('=== explore_crop_distribution ===')
for crop in ['tea', 'wheat', 'rice', 'maize', 'banana', 'coffee', 'potato', 'cassava']:
    rainfall, elevation = explore_crop_distribution(MD_agric_df, crop)
    print(f'  {crop:<10} → Rainfall: {rainfall:,.1f} mm  |  Elevation: {elevation:,.1f} m')

In [ ]:
def analyse_soil_fertility(df):
    """
    Returns the mean soil fertility grouped by soil type.

    Parameters
    ----------
    df : pd.DataFrame — the agricultural dataset

    Returns
    -------
    pd.Series : mean Soil_fertility per Soil_type, sorted ascending

    Example
    -------
    >>> analyse_soil_fertility(MD_agric_df)
    Soil_type
    Loamy    0.5859
    ...
    """
    return df.groupby('Soil_type')['Soil_fertility'].mean().sort_values()


# --- Test & Visualise ---
soil_fertility = analyse_soil_fertility(MD_agric_df)
print('=== Soil Fertility by Soil Type ===')
print(soil_fertility.round(4))

fig, ax = plt.subplots(figsize=(9, 4))
soil_fertility.plot(kind='barh', ax=ax, color=PALETTE[1], alpha=0.85)
ax.set_title('Mean Soil Fertility by Soil Type (0=infertile, 1=very fertile)')
ax.set_xlabel('Mean Soil Fertility')
ax.axvline(soil_fertility.mean(), color=PALETTE[2], linestyle='--',
           linewidth=1.5, label=f'Overall mean: {soil_fertility.mean():.3f}')
ax.legend()
plt.tight_layout()
plt.savefig('chart3_soil_fertility.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
def climate_geography_influence(df, column):
    """
    Analyses how climate and geography vary across a given grouping column.

    Parameters
    ----------
    df     : pd.DataFrame — the agricultural dataset
    column : str          — column to group by (e.g. 'Crop_type', 'Location')

    Returns
    -------
    pd.DataFrame : grouped means of Elevation, Min_temperature_C,
                   Max_temperature_C, and Rainfall
    """
    return df.groupby(column)[[
        'Elevation', 'Min_temperature_C', 'Max_temperature_C', 'Rainfall'
    ]].mean().round(2)


# --- Test by crop type ---
print('=== Climate & Geography by Crop Type ===')
climate_by_crop = climate_geography_influence(MD_agric_df, 'Crop_type')
print(climate_by_crop.to_string())

# --- Visualise ---
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
metrics = ['Elevation', 'Min_temperature_C', 'Max_temperature_C', 'Rainfall']

for ax, metric in zip(axes.flatten(), metrics):
    climate_by_crop[metric].sort_values().plot(
        kind='barh', ax=ax, color=PALETTE[0], alpha=0.85
    )
    ax.set_title(f'Mean {metric} by Crop Type')
    ax.set_xlabel(metric)

plt.suptitle('Climate & Geography Influence by Crop Type', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('chart4_climate_geography.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
def find_ideal_fields(df):
    """
    Identifies the crop type with the most above-average yield fields —
    i.e. the top-performing crop in Maji Ndogo.

    Parameters
    ----------
    df : pd.DataFrame — the agricultural dataset

    Returns
    -------
    str : name of the top-performing crop type
    """
    avg_yield = df['Standard_yield'].mean()
    above_avg = df[df['Standard_yield'] > avg_yield]
    top_crop  = above_avg.groupby('Crop_type')['Standard_yield'].count() \
                         .sort_values(ascending=False)
    return top_crop.index[0]


top_crop = find_ideal_fields(MD_agric_df)
print(f'🏆 Top-performing crop in Maji Ndogo: {top_crop.upper()}')
print(f'   (Most fields with above-average Standard_yield)')

In [ ]:
def find_good_conditions(df, crop_type):
    """
    Filters fields for a given crop type that meet optimal growing conditions:
      - Above-average Standard_yield
      - Average temperature between 12°C and 15°C
      - Pollution level below 0.0001

    Parameters
    ----------
    df        : pd.DataFrame — the agricultural dataset
    crop_type : str          — crop to filter (e.g. 'tea')

    Returns
    -------
    pd.DataFrame : filtered fields meeting all conditions
    """
    avg_yield = df['Standard_yield'].mean()
    filtered  = df[
        (df['Crop_type']      == crop_type)  &
        (df['Standard_yield'] >  avg_yield)  &
        (df['Ave_temps']      >= 12)         &
        (df['Ave_temps']      <= 15)         &
        (df['Pollution_level'] < 0.0001)
    ]
    return filtered


# --- Test ---
good_tea_fields = find_good_conditions(MD_agric_df, 'tea')
print(f'✅ Optimal tea fields found: {len(good_tea_fields)} fields')
print(f'   Shape: {good_tea_fields.shape}')
print(f'\nSample optimal fields:')
print(good_tea_fields[['Elevation','Rainfall','Ave_temps','Pollution_level','Standard_yield']].head())

---
## 6. Advanced Analysis — df.query() & Comparisons

In [ ]:
# Efficient filtering using SQL-like syntax with df.query()
high_yield_loamy = MD_agric_df.query('Standard_yield > 0.5 and Soil_type == "Loamy"')
print(f'High-yield Loamy fields: {len(high_yield_loamy):,}')

# Multi-soil comparison
fertile_soils = ['Loamy', 'Sandy', 'Silt']
fertile_fields = MD_agric_df.query('Soil_type in @fertile_soils')
print(f'Fields in fertile soil types: {len(fertile_fields):,}')

# Crop-wise yield summary using groupby
crop_summary = MD_agric_df.groupby('Crop_type').agg(
    Fields       = ('Standard_yield', 'count'),
    Mean_yield   = ('Standard_yield', 'mean'),
    Mean_rain    = ('Rainfall',       'mean'),
    Mean_elev    = ('Elevation',      'mean'),
    Mean_poll    = ('Pollution_level','mean'),
).round(3).sort_values('Mean_yield', ascending=False)

print('\n=== Full Crop Performance Summary ===')
print(crop_summary.to_string())

In [ ]:
# ── CHART 5: Correlation heatmap ──
numeric_cols = [
    'Elevation','Rainfall','Min_temperature_C','Max_temperature_C',
    'Ave_temps','Soil_fertility','pH','Pollution_level',
    'Plot_size','Standard_yield','Annual_yield'
]
corr = MD_agric_df[numeric_cols].corr()

fig, ax = plt.subplots(figsize=(11, 9))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdYlGn',
            center=0, linewidths=0.4, ax=ax, cbar_kws={'shrink':0.8})
ax.set_title('Feature Correlation Matrix — Maji Ndogo Agricultural Data', pad=14)
plt.tight_layout()
plt.savefig('chart5_correlation.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Chart 5 saved')

In [ ]:
# ── CHART 6: Boxplots — yield by crop type ──
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

crop_order = MD_agric_df.groupby('Crop_type')['Standard_yield'].median() \
                         .sort_values(ascending=False).index

MD_agric_df.boxplot(
    column='Standard_yield', by='Crop_type',
    ax=axes[0], vert=False,
    boxprops=dict(color=PALETTE[1]),
    medianprops=dict(color=PALETTE[0], linewidth=2),
)
axes[0].set_title('Standard Yield Distribution by Crop')
axes[0].set_xlabel('Standard Yield')
plt.suptitle('')

soil_order = MD_agric_df.groupby('Soil_type')['Standard_yield'].median() \
                          .sort_values(ascending=False).index
MD_agric_df.boxplot(
    column='Standard_yield', by='Soil_type',
    ax=axes[1], vert=False,
    boxprops=dict(color=PALETTE[0]),
    medianprops=dict(color=PALETTE[2], linewidth=2),
)
axes[1].set_title('Standard Yield Distribution by Soil Type')
axes[1].set_xlabel('Standard Yield')
plt.suptitle('Yield Distributions — Maji Ndogo', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.savefig('chart6_boxplots.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 7. Production Module — FieldDataProcessor

The `field_data_processor.py` module in this repo implements a production-quality Python class that wraps the entire data pipeline: SQL ingestion → column correction → crop type standardisation → weather station data merge. It uses `data_ingestion.py` as a backend utility and includes full logging support.

**Usage:**
```python
from field_data_processor import FieldDataProcessor

config_params = {
    'db_path':           'sqlite:///Maji_Ndogo_farm_survey_small.db',
    'sql_query':         'SELECT * FROM geographic_features',
    'columns_to_rename': {'Rainfall': 'Min_temperature_C'},
    'values_to_rename':  {'cassaval': 'cassava', 'teaa': 'tea'},
    'weather_mapping_csv': 'https://path/to/weather_data.csv'
}
processor = FieldDataProcessor(config_params, logging_level='INFO')
processor.process()
df_clean = processor.df
```

---
## 8. Data Validation Tests

The `validate_data.py` module contains automated tests to verify data quality after processing — a professional data engineering practice.

In [ ]:
# Run validation checks directly on our cleaned DataFrame
valid_crop_types = ['cassava','tea','wheat','potato','banana','coffee','rice','maize']

checks = {
    'Column count == 17':          MD_agric_df.shape[1] == 17,
    'No negative elevation':       not (MD_agric_df['Elevation'] < 0).any(),
    'Exactly 8 crop types':        len(MD_agric_df['Crop_type'].unique()) == 8,
    'All crop types valid':        all(c in valid_crop_types for c in MD_agric_df['Crop_type'].unique()),
    'No missing Standard_yield':   MD_agric_df['Standard_yield'].isnull().sum() == 0,
    'Pollution level in [0,1]':    MD_agric_df['Pollution_level'].between(0,1).all(),
    'Soil_fertility in [0,1]':     MD_agric_df['Soil_fertility'].between(0,1).all(),
}

print('=== Data Validation Results ===')
all_passed = True
for check, result in checks.items():
    status = '✅ PASS' if result else '❌ FAIL'
    if not result:
        all_passed = False
    print(f'  {status}  {check}')

print(f'\n{"All checks passed! ✅" if all_passed else "Some checks failed ❌"}')

---
## 9. Key Findings & Agricultural Insights

In [ ]:
top_crop       = find_ideal_fields(MD_agric_df)
best_soil      = analyse_soil_fertility(MD_agric_df).idxmax()
rain_elevation = explore_crop_distribution(MD_agric_df, 'tea')
optimal_fields = find_good_conditions(MD_agric_df, top_crop)

print('=' * 65)
print('    MAJI NDOGO AGRICULTURAL ANALYSIS — KEY FINDINGS')
print('=' * 65)
print(f"""
🌾 TOP CROP PERFORMANCE
   • Top-performing crop (most above-avg yield fields): {top_crop.upper()}
   • Optimal {top_crop} fields meeting all conditions: {len(optimal_fields)}
   • Optimal conditions: Ave_temp 12–15°C, Pollution < 0.0001

🌱 SOIL INSIGHTS
   • Most fertile soil type: {best_soil}
   • Silt and Volcanic soils outperform sandy and rocky types
   • Soil fertility weakly correlated with yield — location matters more

☔ CLIMATE & GEOGRAPHY
   • Tea thrives at high elevation (~775m) with high rainfall (~1,535mm)
   • Rice & banana prefer low elevation with heavy rainfall
   • Maize & potato grow well at mid-elevation with lower rainfall
   • Temperature range is narrow across crops — pollution is a bigger yield driver

⚠️ POLLUTION IMPACT
   • Pollution level is negatively correlated with Standard_yield
   • Fields with pollution > 0.5 show measurably lower yields across all crops
   • Low-pollution zones should be prioritised for automated farming deployment

💡 DEPLOYMENT RECOMMENDATIONS
   • Priority crop for automation: {top_crop.upper()} in low-pollution, mid-elevation zones
   • Silt and Volcanic soil regions offer best fertility for expansion
   • High-rainfall areas suit tea, coffee, rice, and banana cultivation
   • Low-rainfall areas better suited to wheat, maize, and potato
""")
print('=' * 65)

---
## 10. Conclusion

| Step | Technique Used |
|------|----------------|
| Data Ingestion | SQL JOIN across 4 tables via SQLAlchemy + Pandas |
| Data Cleaning | Column swap correction, spelling standardisation, abs() for elevation |
| EDA | Distribution plots, scatter plots, boxplots, grouped aggregations |
| Analysis | 5 reusable analytical functions with full docstrings |
| Advanced Filtering | `df.query()` with variable injection |
| Correlation Analysis | Seaborn heatmap across 11 numeric features |
| Validation | Automated data quality checks |
| Production Module | OOP `FieldDataProcessor` class with logging & error handling |

**Repository structure:**
```
maji-ndogo-agricultural-analysis/
├── maji_ndogo_agricultural_analysis.ipynb   ← Main notebook (this file)
├── data_ingestion.py                        ← DB engine + SQL query + CSV loader utilities
├── field_data_processor.py                  ← OOP data pipeline class
├── validate_data.py                         ← Automated data quality tests
├── Maji_Ndogo_farm_survey_small.db          ← SQLite source database
└── README.md                                ← Project documentation
```

---
**Author:** Oluwatobi Victoria Babalola  
**Contact:** bablotobi@gmail.com  
**LinkedIn:** [oluwatobi-babalola-victoria](https://www.linkedin.com/in/oluwatobi-babalola-victoria/)  
**GitHub:** [TechieBbt](https://github.com/TechieBbt)  
**Portfolio:** [bbt-portfolio.vercel.app](https://bbt-portfolio.vercel.app)